In [1]:
import pandas as pd
import numpy as np
import json
from sklearn.ensemble import RandomForestClassifier
bio= '/Users/filippofocaccia/Desktop/adlm-knee-osteoarthritis/data/tabular/Enrollees.txt'
file_path = '/Users/filippofocaccia/Desktop/adlm-knee-osteoarthritis/data/tabular/AllClinical00.txt'
second_file_path = '/Users/filippofocaccia/Desktop/adlm-knee-osteoarthritis/data/tabular/kxr_sq_bu00.txt'
third_file_path = '/Users/filippofocaccia/Desktop/adlm-knee-osteoarthritis/data/tabular/Outcomes99.txt'
fourth_file_path = '/Users/filippofocaccia/Desktop/adlm-knee-osteoarthritis/data/tabular/kxr_qjsw_duryea00.txt'
bio_df = pd.read_csv(bio, sep='|')
df = pd.read_csv(file_path, sep='|')
df_2 = pd.read_csv(second_file_path, sep='|')
df_3 = pd.read_csv(third_file_path, sep='|')
df_4 = pd.read_csv(fourth_file_path, sep='|')
#so we need to filter the symptomes variables from the first file,
#the structure variables from the second file and the kl grade too
# the surgery variables from the third file


/Users/filippofocaccia/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
clinical= df.copy()
clinical2= df_2.copy()
clinical3= df_3.copy()
clinical4= bio_df.copy()

In [13]:
# Filter the dataframe to keep only the columns specified in VARIABLE_DESCRIPTIONS
# Load the JSON file
with open('../variables_t.json', 'r') as file:
	variables = json.load(file)
columns_to_keep_first= ['ID']
columns_to_keep_second= ['ID','SIDE']
columns_to_keep_third= ['id']
columns_to_keep_fourth= ['ID']

# Extract the columns to keep from the allclinicall00 file for the symptoms (WOMAC and KOOS)
columns_to_keep_first.extend(variables["VARIABLES"]["ALL_R"]["SYMPTOMS"])
columns_to_keep_first.extend(variables["VARIABLES"]["ALL_L"]["SYMPTOMS"][:2])
columns_to_keep_first.extend(variables["VARIABLES"]["ALL_L"]["BIO"][:2])
#extract the columns to keep from the kxr_sq_bu00 file for the structure and kl grade
columns_to_keep_second.extend(variables["VARIABLES"]["ALL_R"]["STRUCTURE"])
columns_to_keep_second.extend(variables["VARIABLES"]["ALL_R"]["KL_GRADE"])

#extract the columns to keep from the outcomes99 file for the surgery
columns_to_keep_third.extend(variables["VARIABLES"]["ALL_R"]["SURGERY"])
columns_to_keep_third.extend(variables["VARIABLES"]["ALL_L"]["SURGERY"])

columns_to_keep_fourth.extend(variables["VARIABLES"]["ALL_R"]["BIO"][2:])
clinical = clinical[columns_to_keep_first]
clinical2 = clinical2[columns_to_keep_second]
clinical3 = clinical3[columns_to_keep_third]
clinical4 = clinical4[columns_to_keep_fourth]

In [14]:
clinical4['P02SEX'] = clinical4['P02SEX'].apply(lambda x: 1 if x == '1: Male' else 2)
#1 for male patients and 2 for female patients

In [18]:
#abdominal circumference
clinical['V00ABCIRC'] = clinical['V00ABCIRC'].fillna(clinical['V00ABCIRC'].mean())

In [ ]:
#i want for each variable in clinical2 to create two new columns in clinical, one for the right side and one for the left side
for col in clinical2.columns:
    if col != 'ID' and col != 'SIDE':
        clinical2[f'{col}_R'] = clinical2.apply(lambda row: row[col] if row['SIDE'] == '1: Right' else None, axis=1)
        clinical2[f'{col}_L'] = clinical2.apply(lambda row: row[col] if row['SIDE'] == '2: Left' else None, axis=1)
        # Drop the original columns
        clinical2.drop(columns=[col], inplace=True)
clinical2.drop(columns=['SIDE'], inplace=True)
clinical2= clinical2.groupby("ID").first().reset_index()


,ID,V00XRJSL_R,V00XRJSL_L,V00XRJSM_R,V00XRJSM_L,V00XRSCFM_R,V00XRSCFM_L,V00XRSCFL_R,V00XRSCFL_L,V00XRSCTM_R,...,V00XROSFM_R,V00XROSFM_L,V00XROSFL_R,V00XROSFL_L,V00XROSTM_R,V00XROSTM_L,V00XROSTL_R,V00XROSTL_L,V00XRKL_R,V00XRKL_L
0,9000099,0.0,2.0,0.0,0.0,0: 0,0: 0,0: 0,2: 2,0: 0,...,0: 0,0: 0,2: 2,2: 2,1: 1,0: 0,1: 1,1: 1,2: 2,3: 3
1,9000296,0.0,0.0,1.0,2.0,0: 0,0: 0,0: 0,0: 0,0: 0,...,0: 0,0: 0,0: 0,0: 0,1: 1,1: 1,0: 0,0: 0,2: 2,3: 3
2,9000622,0.0,0.0,0.0,0.0,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,...,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,1: 1,1: 1
3,9000798,0.0,0.0,1.0,3.0,0: 0,2: 2,0: 0,0: 0,0: 0,...,0: 0,3: 3,0: 0,2: 2,0: 0,2: 2,0: 0,2: 2,1: 1,4: 4
4,9001104,0.0,0.0,2.0,1.0,2: 2,0: 0,0: 0,0: 0,1: 1,...,2: 2,0: 0,0: 0,0: 0,2: 2,0: 0,0: 0,0: 0,3: 3,1: 1


In [21]:
clinical3.rename(columns={'id': 'ID'}, inplace=True)
#now we can merge the three dataframes on the ID column
merged_df = clinical.merge(clinical2, on='ID', how='inner').merge(clinical3, on='ID', how='inner').merge(clinical4, on='ID', how='inner')
merged_df.head()

,ID,V00WOMTSR,V00KOOSKPR,V00KOOSYMR,V00KOOSQOL,V00WOMTSL,V00KOOSKPL,V00AGE,V00ABCIRC,V00XRJSL_R,...,V00XROSFL_L,V00XROSTM_R,V00XROSTM_L,V00XROSTL_R,V00XROSTL_L,V00XRKL_R,V00XRKL_L,V99ERKVSAF,V99ELKVSAF,P02SEX
0,9000099,14.0,77.8,67.9,25.0,0.0,100.0,59,96.8,0.0,...,2: 2,1: 1,0: 0,1: 1,1: 1,2: 2,3: 3,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,1
1,9000296,0.0,100.0,100.0,100.0,0.0,100.0,69,104.3,0.0,...,0: 0,1: 1,1: 1,0: 0,0: 0,2: 2,3: 3,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,1
2,9000622,20.9,75.0,82.1,50.0,0.0,100.0,71,98.9,0.0,...,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,1: 1,1: 1,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,2
3,9000798,0.0,100.0,100.0,43.8,30.0,59.4,56,109.0,0.0,...,2: 2,0: 0,2: 2,0: 0,2: 2,1: 1,4: 4,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,1
4,9001104,33.0,69.4,60.7,37.5,14.0,100.0,72,111.1,0.0,...,0: 0,2: 2,0: 0,0: 0,0: 0,3: 3,1: 1,.: Missing Form/Incomplete Workbook,.: Missing Form/Incomplete Workbook,2


In [22]:
#i want 0 for Missing Form/Incomplete Workbook and 1 for everything else
merged_df['V99ELKVSAF'] = merged_df['V99ELKVSAF'].apply(lambda x: 0 if x == '.: Missing Form/Incomplete Workbook' else 1)
merged_df['V99ERKVSAF']= merged_df['V99ERKVSAF'].apply(lambda x: 0 if x == '.: Missing Form/Incomplete Workbook' else 1)
#let's  see how many missing values we have in each column for the kl
merged_df['V00XRKL_R'] = merged_df['V00XRKL_R'].apply(lambda x: np.nan if x == '.: Missing Form/Incomplete Workbook' else x)
merged_df['V00XRKL_L'] = merged_df['V00XRKL_L'].apply(lambda x: np.nan if x == '.: Missing Form/Incomplete Workbook' else x)

# Replace missing values with the mean for the specified columns in the symptomes
mean_columns = ['V00WOMTSR', 'V00WOMTSL', 'V00KOOSKPL', 'V00KOOSKPR', 'V00KOOSQOL']
for col in mean_columns:
    merged_df[col] = merged_df[col].fillna(round(clinical[col].mean()))

#KL is very much specular between left and right, so we can fill the missing values accordingly
merged_df["V00XRKL_R"] = merged_df["V00XRKL_R"].fillna(merged_df["V00XRKL_L"])
merged_df["V00XRKL_L"] = merged_df["V00XRKL_L"].fillna(merged_df["V00XRKL_R"])

#similarly for the structure variables
for comp in ["JSL", "JSM"]:  # lateral, medial
    merged_df[f"V00XR{comp}_R"] = merged_df[f"V00XR{comp}_R"].fillna(merged_df[f"V00XR{comp}_L"])
    merged_df[f"V00XR{comp}_L"] = merged_df[f"V00XR{comp}_L"].fillna(merged_df[f"V00XR{comp}_R"])


In [23]:
#after all these operations we can drop the remaining na values given its only one row
merged_df.dropna(inplace=True)

In [24]:
#as you can see no missing values anymore
merged_df.isna().sum().sort_values(ascending=False).head(10)

ID             0
V00XROSTM_R    0
V00XRSCTL_R    0
V00XRSCTL_L    0
V00XROSFM_R    0
V00XROSFM_L    0
V00XROSFL_R    0
V00XROSFL_L    0
V00XROSTM_L    0
V00WOMTSR      0
dtype: int64

In [25]:
# i need to map the kl grades to numerical values ['2: 2', '1: 1', '3: 3', '0: 0', '4: 4'] to [2,1,3,0,4]
kl_mapping = {'0: 0': 0, '1: 1': 1, '2: 2': 2, '3: 3': 3, '4: 4': 4}
merged_df['V00XRKL_R'] = merged_df['V00XRKL_R'].map(kl_mapping)
merged_df['V00XRKL_L'] = merged_df['V00XRKL_L'].map(kl_mapping)


In [26]:
c = [
    "V00XRSCFM_L","V00XRSCFM_R","V00XRSCFL_L","V00XRSCFL_R",
    "V00XRSCTM_L","V00XRSCTM_R","V00XRSCTL_L","V00XRSCTL_R",
    "V00XROSFM_L","V00XROSFM_R","V00XROSFL_L","V00XROSFL_R",
    "V00XROSTM_L","V00XROSTM_R","V00XROSTL_L","V00XROSTL_R",
]
c_j = ["V00XRJSM_L","V00XRJSM_R","V00XRJSL_L","V00XRJSL_R"]

MISSING = ".: Missing Form/Incomplete Workbook"
CODE_MAP = {"0: 0": 0, "1: 1": 1, "2: 2": 2, "3: 3": 3}

def clean_xr_columns(df: pd.DataFrame, cols, code_map=CODE_MAP) -> None:
    cols = list(cols)
    present = [col for col in cols if col in df.columns]
    missing = [col for col in cols if col not in df.columns]

    if missing:
        print(f"Warning: {len(missing)} columns not found: {missing}")

    if not present:
        return

    # Replace the sentinel with NA (vectorized)
    df[present] = df[present].replace(MISSING, pd.NA)

    # Map the "k: k" strings to ints (vectorized). Anything else becomes NA.
    df[present] = df[present].replace(code_map)

    # Cast to pandas nullable integer
    df[present] = df[present].astype("Int64")

# Clean both sets
clean_xr_columns(merged_df, c)
clean_xr_columns(merged_df, c_j, code_map={}) 

In [27]:
merged_df.isna().sum().sort_values(ascending=False).head(20)

V00XRSCTM_R    1758
V00XRSCFM_R    1758
V00XRSCFL_R    1758
V00XRSCTL_R    1758
V00XRSCTL_L    1748
V00XRSCFM_L    1748
V00XRSCFL_L    1748
V00XRSCTM_L    1748
V00XROSTL_R    1741
V00XROSTM_R    1741
V00XROSFL_R    1741
V00XROSFM_R    1741
V00XROSTL_L    1732
V00XROSTM_L    1732
V00XROSFM_L    1732
V00XROSFL_L    1732
V00XRKL_L         0
V00XRKL_R         0
V99ERKVSAF        0
V99ELKVSAF        0
dtype: int64

In [28]:
merged_df

,ID,V00WOMTSR,V00KOOSKPR,V00KOOSYMR,V00KOOSQOL,V00WOMTSL,V00KOOSKPL,V00AGE,V00ABCIRC,V00XRJSL_R,...,V00XROSFL_L,V00XROSTM_R,V00XROSTM_L,V00XROSTL_R,V00XROSTL_L,V00XRKL_R,V00XRKL_L,V99ERKVSAF,V99ELKVSAF,P02SEX
0,9000099,14.0,77.8,67.9,25.0,0.0,100.0,59,96.8,0,...,2,1,0,1,1,2,3,0,0,1
1,9000296,0.0,100.0,100.0,100.0,0.0,100.0,69,104.3,0,...,0,1,1,0,0,2,3,0,0,1
2,9000622,20.9,75.0,82.1,50.0,0.0,100.0,71,98.9,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,1,1,0,0,2
3,9000798,0.0,100.0,100.0,43.8,30.0,59.4,56,109.0,0,...,2,0,2,0,2,1,4,0,0,1
4,9001104,33.0,69.4,60.7,37.5,14.0,100.0,72,111.1,0,...,0,2,0,0,0,3,1,0,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4502,9999365,27.0,58.3,82.1,18.8,26.0,72.2,56,109.3,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,0,0,0,0,1
4503,9999510,0.0,100.0,100.0,68.8,11.8,83.3,50,97.5,0,...,0,1,1,0,0,1,3,0,0,1
4504,9999862,0.0,97.2,100.0,93.8,0.0,97.2,61,95.6,0,...,0,1,1,1,0,2,2,0,0,2
4505,9999865,0.0,100.0,100.0,100.0,0.0,100.0,61,95.5,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,0,1,0,0,2


In [29]:
#save the final dataset
merged_df.to_csv('../csv/clinical00_cleaned.csv', index=False)